In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType
from datetime import datetime

dbutils.widgets.text("bucket", "DataStagingBucket", "S3 Bucket")
dbutils.widgets.text("env", "prod", "Environment")

# 1. Define the path to CSV file
s3_bucket = dbutils.widgets.get("bucket")
env = dbutils.widgets.get("env")

# -------------------- csv path, target table name and schema -------------------
file_path = f"s3a://{s3_bucket}/edw_exports/{env}/mismatchedpagecounts__20260312.csv"
new_table_name = "default.factmismatchpagecount"

# schema mapping CSV's columns
csv_schema = StructType([
    StructField("visualrequestfilenumber", StringType(), True),
    StructField("recordspagecount", IntegerType(), True),
    StructField("noofpagesintherequest", IntegerType(), True),
    StructField("finalcount", IntegerType(), True),
    StructField("recordspagecount_old", IntegerType(), True),
    StructField("noofpagesintherequest_old", IntegerType(), True),
    StructField("finalcount_old", IntegerType(), True)
])

# 2. Read the CSV into a DataFrame
csv_df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("ignoreTrailingWhiteSpace", "true")
    .schema(csv_schema)
    .load(file_path)
)

# 3. Save into a temp view
csv_df.createOrReplaceTempView("temp_mismatchedpagecount")

# ==========================================================
query = """
    SELECT 
        a.visualrequestfilenumber,
        a.lanpagecount,
        a.noofpagesintherequest,
        a.recordspagecount,
        a.modpagecount,
        -- t.recordspagecount as recordspagecount_new,
        -- t.noofpagesintherequest as noofpagesintherequest_new,
        -- t.finalcount as finalcount_new,
        t.recordspagecount_old,
        t.noofpagesintherequest_old,
        t.finalcount_old,
        CASE
            WHEN a.recordspagecount >= t.finalcount_old THEN 'recordspagecount'
            WHEN a.modpagecount = t.finalcount_old THEN 'modpagecount'
            WHEN a.lanpagecount = t.finalcount_old and a.recordspagecount = 0 THEN 'lanpagecount'
            WHEN a.recordspagecount < t.finalcount_old and a.modpagecount <> t.finalcount_old and a.lanpagecount <> t.finalcount_old and (a.recordspagecount = 0 or t.finalcount_old - a.recordspagecount > 100) THEN 'finalcount_old'
            ELSE 'recordspagecount'
        END AS activefield
    FROM 
        temp_mismatchedpagecount t
    INNER JOIN 
        default.materialized_viw_allopenclosed a ON a.visualrequestfilenumber = t.visualrequestfilenumber
    ORDER BY a.visualrequestfilenumber
"""

# Store the result back into a new DataFrame
joined_df = spark.sql(query)

print(f"Number of mismatch pagecount: {joined_df.count()}")

# Display the results or write them back out
display(joined_df)

# 4. Save into the new Delta table
# (joined_df.write
#     .format("delta")
#     .mode("overwrite") # Overwrites the table if it already exists
#     .saveAsTable(new_table_name)
# )